# SerendibAI Gemma 4 E4B – Colab inference dashboard

This notebook runs the official `google/gemma-4-E4B-it` checkpoint with the Hugging Face Transformers `image-text-to-text` pipeline in text-only chat mode. It presents a temporary public Gradio link for manual inference.

Before running, select **Runtime → Change runtime type → GPU**. A T4 (16 GB) or better is required. You must first accept the Gemma terms on Hugging Face. Add a read token named `HF_TOKEN` in **Colab Secrets** and grant this notebook access to it, or enter the token at the hidden prompt. The token is exported only to this Colab runtime; it is not written into the notebook or saved to Git.

In [ ]:
!nvidia-smi
!pip install -q --upgrade "transformers>=5.3.0" accelerate gradio

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except ImportError:
    hf_token = None
except userdata.SecretNotFoundError:
    hf_token = None

if not hf_token:
    hf_token = getpass("Hugging Face token (input stays hidden): " )

if not hf_token:
    raise RuntimeError("An HF_TOKEN with Gemma access is required.")

# Makes the token available to Transformers and Hugging Face Hub for this runtime only.
os.environ["HF_TOKEN"] = hf_token

In [ ]:
import torch
from transformers import pipeline

MODEL_ID = "google/gemma-4-E4B-it"

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. Select a GPU runtime and rerun the notebook.")

generator = pipeline(
    task="image-text-to-text",
    model=MODEL_ID,
    token=hf_token,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    model_kwargs={"attn_implementation": "sdpa"},
)
print(f"Loaded {MODEL_ID} on {torch.cuda.get_device_name(0)}.")

In [ ]:
import gradio as gr

DEFAULT_SYSTEM_PROMPT = "You are a helpful, concise assistant."

def extract_answer(result, prompt):
    generated = result[0]["generated_text"]
    if isinstance(generated, list):
        return generated[-1]["content"]
    return generated[len(prompt):] if generated.startswith(prompt) else generated

def respond(message, history, system_prompt, temperature, max_tokens):
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(
        {"role": item["role"], "content": item["content"]}
        for item in history
        if item["role"] in {"user", "assistant"}
    )
    messages.append({"role": "user", "content": message})
    prompt = generator.processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    result = generator(
        text=prompt,
        do_sample=temperature > 0,
        temperature=max(float(temperature), 0.01),
        max_new_tokens=int(max_tokens),
    )
    return extract_answer(result, prompt)

with gr.Blocks(title="SerendibAI Gemma 4 E4B") as demo:
    gr.Markdown("# SerendibAI Gemma 4 E4B\nTransformers pipeline on a Colab GPU. The shared link is temporary and public—do not enter secrets or customer data.")
    with gr.Accordion("Generation settings", open=False):
        system_prompt = gr.Textbox(value=DEFAULT_SYSTEM_PROMPT, label="System prompt", lines=3)
        with gr.Row():
            temperature = gr.Slider(0, 1.5, value=0.2, step=0.05, label="Temperature")
            max_tokens = gr.Slider(16, 512, value=160, step=16, label="Max new tokens")
    gr.ChatInterface(
        fn=respond,
        additional_inputs=[system_prompt, temperature, max_tokens],
        type="messages",
        examples=["Hello! Please introduce yourself in Sinhala.", "What can you help me with?"],
    )

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)